In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import PSNR,SSIM
import torch




# ── 核心函数 ───────────────────────────────────────────────────────────────────
def fft_lowpass_filter(img: np.ndarray, c: float):
    """
    对图像做 FFT，仅保留 100c% 的低频成分，高频置零，再逆变换。

    Parameters
    ----------
    img : np.ndarray
        输入图像，形状 (H, W) 灰度 或 (H, W, 3) RGB，dtype uint8。
    c   : float
        保留低频的比例，取值范围 (0, 1]。
        例如 c=0.1 表示只保留 10% 的低频，90% 高频置零。

    Returns
    -------
    filtered : np.ndarray
        经低通滤波后的图像，dtype uint8，形状与 img 相同。
    """
    assert 0 < c <= 1, "c 必须在 (0, 1] 之间"

    def _filter_channel(channel: np.ndarray) -> np.ndarray:
        H, W = channel.shape

        # 1. FFT & 中心化
        f_shift = np.fft.fftshift(np.fft.fft2(channel))

        # 2. 构造低通掩膜（保留中心 c*H × c*W 的矩形区域）
        mask = np.zeros((H, W), dtype=np.float32)
        rH = int(round(H * c / 2))   # 半径（行方向）
        rW = int(round(W * c / 2))   # 半径（列方向）
        cH, cW = H // 2, W // 2      # 频谱中心
        mask[cH - rH : cH + rH, cW - rW : cW + rW] = 1.0

        # 3. 应用掩膜，逆变换
        filtered_channel = np.fft.ifft2(np.fft.ifftshift(f_shift * mask))
        return np.clip(np.abs(filtered_channel), 0, 255).astype(np.uint8)

    # 灰度 or 彩色分别处理
    if img.ndim == 2:
        filtered = _filter_channel(img)
    else:
        filtered = np.stack([_filter_channel(img[:, :, ch]) for ch in range(img.shape[2])], axis=-1)

    return filtered


# ── 可视化 + 评价 ──────────────────────────────────────────────────────────────
def visualize_and_evaluate(img: np.ndarray, c: float):
    """
    调用 fft_lowpass_filter，展示三张子图并打印 PSNR / SSIM。

    子图布局：
        [原图]  [低通滤波图]  [差值图(原图-滤波图)]
    """
    filtered = fft_lowpass_filter(img, c)

    # 差值图（绝对值，拉伸到 0‑255 方便观察）
    diff = np.abs(img.astype(np.int16) - filtered.astype(np.int16)).astype(np.uint8)

    # ── 指标 ──
    def to_tensor(img_np):
        return torch.from_numpy(img_np).float() / 255.0

    # 你的 fft 函数输出的是 uint8 numpy，转一下再传给 PSNR/SSIM
    # img_tensor      = to_tensor(img_array)       # 原图
    filtered_tensor = to_tensor(filtered)        # fft_lowpass_filter 的输出

    # psnr_val = psnr(torch.tensor(img), filtered_tensor)
    # ssim_val = ssim(torch.tensor(img), filtered_tensor)
   
    print(f"c = {c:.2f}  ({c*100:.1f}%)")
    # print(f"  PSNR : {psnr_val:.4f} dB")
    # print(f"  SSIM : {ssim_val:.4f}")

    # ── 绘图 ──
    cmap = "gray" if img.ndim == 2 else None
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img,      cmap=cmap, vmin=0, vmax=255)
    axes[0].set_title("(Original)")
    axes[0].axis("off")

    axes[1].imshow(filtered, cmap=cmap, vmin=0, vmax=255)
    axes[1].set_title(f"(c={c:.2f})\n {c*100:.1f}% ")
    axes[1].axis("off")

    axes[2].imshow(diff,     cmap="hot", vmin=0, vmax=255)
    # axes[2].set_title(f"差值图 |原图 − 滤波图|\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    
    return to_tensor(img).permute(2,0,1), filtered_tensor.permute(2,0,1)

# ── 使用示例 ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    from PIL import Image
    
    pnsr = PSNR()
    ssim = SSIM()


    idx = 801
    pic = Image.open(f"/home/liuy/data/raw/DIV2K/DIV2K_valid_HR/{idx:04d}.png").convert("RGB")
    pic = np.array(pic)
    img, filter = visualize_and_evaluate(pic, 0.8)
    print(pnsr(img, filter), ssim(img, filter))
    # print(img[0,0,0], filter[0,0,0])

In [ ]:
from models.models import EDSR_MSFNO
from utils._utils import count_parameters
from omegaconf import OmegaConf

path = "config\EDSR_MSFNO.yaml"
config = OmegaConf.load(path)
m = EDSR_MSFNO.EDSR_MSFNO(config)
print(count_parameters(m.encoder))

ValueError: EDSR_CIPFNO already registered in model